# Trigger wave thresholds

In [1]:
import immunowave as iw
import numpy as np
import jax
import jax.numpy as jnp
import diffrax as dx

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from multiprocessing import Pool
import multiprocessing as mp
from functools import partial

import pickle

jax.config.update("jax_enable_x64", True)
jax.config.update('jax_platform_name', 'cpu')

from hill_function_utils import tissue_response, single_cell_response, compute_wave_threshold, compute_cell_threshold

import warnings
warnings.simplefilter("ignore")

from immunowave_paper_utils import style_axes, colors, fontsize, linewidth, rc_params
from diffrax import SaveAt

In [2]:
%matplotlib qt

In [172]:
linewidth = 4
fontsize = 24
markersize = 24
markeredgewidth = 4
rc_params['axes.linewidth'] = linewidth
rc_params['font.size'] = fontsize
mpl.rcParams.update(rc_params)
mpl.rcParams['pdf.fonttype'] = 42

In [4]:
"""ok, so multiprocessing with jax only works if you use a 
non-default start method within the multiprocessing library. 
Here I use forkserver. An annoying corollary of forkserver 
(and also spawn) is that in jupyter notebooks, functions 
called within a Pool() need to be imported and can't be
defined in another cell. So I moved the functions to a
python file in this same directory."""
mp.set_start_method('forkserver')

## Model definition

Define the state and model for model 4 from the text.

In [5]:
def Bc_cell_theory(n, KD, gamma):
    return (1 - 1/n)*(1/n)**(1/(n-1))*(KD * gamma) ** (n/(n-1))

def Bc_wave_theory(n, KD, gamma, D, a):
    return 2*(1 - 2/(n+1))**(1/2)*(KD * gamma) ** (n/(n-1))*np.sqrt(D / gamma) / a

In [6]:
class State4(iw.State):
    A: iw.ScalarField
    B: iw.ScalarField
        
        
class HillModel(iw.Model):
    KD: float
    n: int
    η: float
    ξ: float
    λ: float
    μ: float
    D: float=1.0
    gamma: float=1.0

    @jax.jit
    def __call__(self, t, state, args=None):
        # unpack field variables
        A, B = state.A, state.B
        # unpack parameters
        KD, n, η, ξ, λ, μ, D, gamma = self.KD, self.n, self.η, self.ξ, self.λ, self.μ, self.D, self.gamma
        # define PDE
        An = A.binop(n, jnp.power)
        tmp = An + KD ** n
        tmp = tmp.binop(-1, jnp.power)
        hill_term = An * tmp
        dAdt = D * A.laplacian(bc="neumann") + hill_term - gamma * A + η * B
        dBdt = ξ * B.laplacian(bc="neumann") + λ * B * (1 - B) - μ * A * B
        return State4(dAdt, dBdt)
    
    
#@jax.jit
def response(B0, KD, t_max, hill_coefficient=2):
    model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0)
    state = State4(
        A=iw.ScalarField(shape, lb, h, 0),
        B=iw.ScalarField(
            shape, lb, h, fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=1)
        ),
    )
    solution = iw.solve(
        model,
        state,
        t0=0,
        t1=t_max,
        t=jnp.array([t_max]),
        **kwargs,
    )
    return np.sum(solution.ys.A.values[-1] * solution.ys.A.h)
    # return solution.evaluate(t_final).A.integral() / L
    

def find_B0(final_mean_A, B0s):
    this_id = np.where(np.diff(final_mean_A) == np.max(np.diff(final_mean_A)))[0][0]
    B0c = 0.5 * (B0s[this_id] + B0s[this_id + 1])
    uncertainty = 0.5 * (B0s[this_id+1] - B0s[this_id])
    return B0c, uncertainty


"""these functions are now imported"""
# def tissue_response(B0, KD, t_max, hill_coefficient=2, D=1, gamma=1):
#     model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, D=D, gamma=gamma)
#     state = State4(
#         A=iw.ScalarField(shape, lb, h, 0),
#         B=iw.ScalarField(
#             shape, lb, h, fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=1)
#         ),
#     )
#     solution = iw.solve(
#         model,
#         state,
#         t0=0,
#         t1=t_max,
#         t=jnp.array([t_max]),
#         **kwargs,
#     )
    
#     tissue_response = np.sum(solution.ys.A.values[-1] * solution.ys.A.h)
    
#     return tissue_response

    
# def single_cell_response(B0, KD, t_max, hill_coefficient=2, gamma=1.0):
#     """single cell response"""
#     model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, D=1.0, gamma=gamma)
#     state = State4(
#         A=iw.ScalarField((1,), lb, 1, 0),
#         B=iw.ScalarField(
#             (1,), lb, 1, fn=lambda x: B0 * jnp.array(x == 0).astype('float')
#         ),
#     )
#     solution = iw.solve(
#         model,
#         state,
#         t0=0,
#         t1=t_max,
#         t=jnp.array([t_max]),
#         **kwargs,
#     )
        
#     single_cell_response = solution.ys.A.values[-1, int(L // 2)]

#     return single_cell_response


'these functions are now imported'

## Plot critical bacteria concentrations as a function of various parameters for both tissue and single cell

### Vary KDs

In [7]:
L = 100.0
n = 2000#200
lb = [0]
h = L / (n - 1)
shape = (n,)

state = State4(
        A=iw.ScalarField(shape, lb, h, 0),
        #B=iw.ScalarField(
        #    shape, lb, h, fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=1)
        #),
        B=iw.ScalarField(
            shape, lb, h, fn=lambda x: 1 * jax.scipy.stats.norm.pdf(x, loc=L/ 2, scale=0.5)
        ),
    )

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [12]:
plt.figure()
state.B.plot()


In [14]:
state.B.h

1.0

In [13]:
state.B.integral()

Array(1., dtype=float64)

In [10]:
Bc_wave(4, KD, gamma, D, 1)

0.07190718501009843

In [40]:
B0s = np.logspace(-6, -3, 40)
t_max = 500
L = 100.0
n = 2000
hill_coefficient = 3
gamma = 0.02
D = 1.0      
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)

B0c, B0c_uncertainty = compute_wave_threshold(B0s, KD=KD, t_max=t_max, 
                       hill_coefficient=hill_coefficient, D=D, gamma=gamma, shape=shape, 
                       lb=lb, h=h, L=L, **kwargs)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [16]:
B0c_uncertainty

0.00020074191550984583

In [41]:
B0c

2.8882429951316953e-05

In [27]:
KD = 0.19
B0 = 0.25
L = 100.0
n = 2000#200
D = 1.0
gamma = 1.0
hill_coefficient = 4#[2, 3, 4, 5]
shape = (n,)
lb = [0]
h = L / (n - 1)
t_max = 1000
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)

model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, D=D, gamma=gamma)
state = State4(
    A=iw.ScalarField(shape, lb, h, 0),
    B=iw.ScalarField(
        shape, lb, h, fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L / 2, scale=1)
    ),
)

solution = iw.solve(
        model,
        state,
        t0=0,
        t1=t_max,
        **kwargs,
    )



In [26]:
((1 - (1 / hill_coefficient)) * (1 / hill_coefficient) ** (1 / (hill_coefficient - 1))) ** (1 / hill_coefficient)

0.8290746760692316

In [22]:
KD

0.31622776601683794

In [16]:
solution.ys.A.values.shape

(1, 2000)

In [28]:
plt.figure()
plt.plot(solution.ys.A.values[-1])

In [25]:
plt.figure()
u_arr = np.linspace(0, 2, 1000)
f1 = u_arr ** hill_coefficient / (KD ** hill_coefficient + u_arr ** hill_coefficient)
f2 = u_arr
plt.plot(u_arr, f1)
plt.plot(u_arr, f2)

In [13]:
KDs = np.logspace(-2, np.log10(0.15), 20)
KD = KDs[-1]
B0s = np.logspace(-1, -0, 40)
L = 100.0
n = 2000#200
D = 1.0
gamma = 1.0
hill_coefficient = 4#[2, 3, 4, 5]
shape = (n,)
lb = [0]
h = L / (n - 1)
t_max = 100
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5
scale = 0.1

Bc_wave_test, Bc_wave_uncertainties_test = compute_wave_threshold(B0s, KD=KD, t_max=t_max, 
                   hill_coefficient=hill_coefficient, D=D, gamma=gamma, shape=shape, 
                   lb=lb, h=h, L=L, n_iters=n_iters, scale=scale, **kwargs)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [14]:
Bc_wave_test

0.12681612237496448

In [12]:
Bc_wave_test

0.18075361311848898

In [8]:
Bc_wave_test

0.14619729265448267

In [9]:
Bc_wave_theory(hill_coefficient, KD, gamma, D, 1)

0.12346976823446565

In [91]:
KDs[KDs < 0.18]

array([0.01      , 0.01199354, 0.0143845 , 0.01725211, 0.02069138,
       0.02481629, 0.02976351, 0.03569699, 0.04281332, 0.05134833,
       0.06158482, 0.073862  , 0.08858668, 0.10624678, 0.1274275 ,
       0.15283067])

In [10]:
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/KD_sweep_wave_v2.pkl', 'rb') as f:
    KD_sweep_dict = pickle.load(f)
KDs = KD_sweep_dict['KDs']
Bc_wave = KD_sweep_dict['Bc_wave']
Bc_wave[-1]

0.14246512389227012

In [15]:
KDs = np.logspace(-2, np.log10(0.15), 20)
B0s = np.logspace(-4, 0, 40)
t_max = 100
L = 100.0
n = 2000#200
D = 1.0
gamma = 1.0
hill_coefficient = 4#[2, 3, 4, 5]
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5
scale = 0.1

"""tissue"""
print('tissue')
Bc_wave = np.zeros(len(KDs))
Bc_wave_uncertainties = np.zeros(len(KDs))
for i, KD in enumerate(KDs):
    print(f'{i + 1} of {len(KDs)}')
    tissue_theory = 2 * (1 - (2 / (hill_coefficient + 1))) ** 0.5 * KD ** (hill_coefficient / (hill_coefficient - 1))
    B0s = np.linspace(0.5 * tissue_theory, 2 * tissue_theory, 40)
    Bc_wave[i], Bc_wave_uncertainties[i] = compute_wave_threshold(B0s, KD=KDs[i], t_max=t_max, 
                       hill_coefficient=hill_coefficient, D=D, gamma=gamma, shape=shape, 
                       lb=lb, h=h, L=L, n_iters=n_iters, scale=scale, **kwargs)
      


      

tissue
1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

2 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3 of 20


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

19 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

20 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [16]:
KD_sweep_dict = {'KDs': KDs, 'Bc_wave': Bc_wave, 'Bc_wave_uncertainties': Bc_wave_uncertainties}
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/KD_sweep_wave_v3_scale0pt1.pkl', 'wb') as f:
    pickle.dump(KD_sweep_dict, f)

In [17]:
"""plot"""
# load the data
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/KD_sweep_wave_v3_scale0pt1.pkl', 'rb') as f:
    KD_sweep_dict = pickle.load(f)
KDs = KD_sweep_dict['KDs']
Bc_wave = KD_sweep_dict['Bc_wave']

plt.figure(figsize=(8,7))
plt.plot(KDs, Bc_wave, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor=colors['wave'], label='wave numerics')
tissue_theory = 2 * (1 - (2 / (hill_coefficient + 1))) ** 0.5 * KDs ** (hill_coefficient / (hill_coefficient - 1))
plt.plot(KDs, tissue_theory, '-', linewidth=linewidth, color=colors['wave'], label='wave theory')

plt.legend(fontsize=18)
plt.xscale("log")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel("concentration scale \nof positive feedback, $K_D$ (a.u.)", fontsize=fontsize)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()


In [108]:
Bc_wave / tissue_theory

array([1.15457948, 1.15581919, 1.15645203, 1.15483236, 1.15387839,
       1.1539473 , 1.1545289 , 1.1538462 , 1.15389689, 1.15385134,
       1.15384615, 1.15384616, 1.15384616, 1.15384687, 1.15384615,
       1.15384621, 1.15385718, 1.1538468 , 1.15384652, 1.15384621])

### Vary hill coefficients

In [20]:
"""vary hill coefficients"""
KD = 0.01#np.logspace(-2, -0.5, 5)
B0s = np.logspace(-5, -2, 40)
t_max = 100
L = 100.0
n = 2000
hill_coefficients = np.linspace(2, 5, 20)#[2, 3, 4, 5]
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
scale = 0.1
n_iters = 5

"""tissue"""
print('tissue')
Bc_wave = np.zeros(len(hill_coefficients))
Bc_wave_uncertainties = np.zeros(len(hill_coefficients))
for i in range(len(hill_coefficients)):
    print(f'{i + 1} of {len(hill_coefficients)}')
    tissue_theory = 2 * (1 - (2 / (hill_coefficients[i] + 1))) ** 0.5 * KD ** (hill_coefficients[i] / (hill_coefficients[i] - 1))
    B0s = np.linspace(0.5 * tissue_theory, 2 * tissue_theory, 40)
    Bc_wave[i], Bc_wave_uncertainties[i] = compute_wave_threshold(B0s, KD=KD, t_max=t_max, 
                       hill_coefficient=hill_coefficients[i], D=D, gamma=gamma, shape=shape, 
                       lb=lb, h=h, L=L, scale=scale, n_iters=n_iters, **kwargs)
      

hill_coeff_sweep_dict = {'hill_coefficients': hill_coefficients, 'Bc_wave': Bc_wave, 'Bc_wave_uncertainties': Bc_wave_uncertainties}
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/hill_coefficient_sweep_scale0pt1.pkl', 'wb') as f:
    pickle.dump(hill_coeff_sweep_dict, f)

tissue
1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

2 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

KeyboardInterrupt: 

In [25]:
hill_coeff_sweep_dict = {'hill_coefficients': hill_coefficients, 'Bc_wave': Bc_wave, 'Bc_cell': Bc_cell, 'Bc_wave_uncertainties': Bc_wave_uncertainties, 'Bc_cell_uncertainties': Bc_cell_uncertainties}
with open(r'/home/brandon/Documents/Code/immunowave/data/initial-theory-numerics-comparison/hill_coefficient_sweep.pkl', 'wb') as f:
    pickle.dump(hill_coeff_sweep_dict, f)

In [21]:
"""plot"""
ns = np.linspace(2, 5, 100)

# load the data
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/hill_coefficient_sweep_scale0pt1.pkl', 'rb') as f:
    hill_coeff_sweep_dict = pickle.load(f)
hill_coefficients = hill_coeff_sweep_dict['hill_coefficients']
Bc_wave = hill_coeff_sweep_dict['Bc_wave']

plt.figure(figsize=(8,7))
plt.plot(hill_coefficients, Bc_wave, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor=colors['wave'], label='wave numerics')
tissue_theory = 2 * (1 - (2 / (ns + 1))) ** 0.5 * KD ** (ns / (ns - 1))
plt.plot(ns, tissue_theory, '-', linewidth=linewidth, color=colors['wave'], label='wave theory')


plt.legend(fontsize=18)
plt.xscale("linear")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel("hill coefficient, $n$", fontsize=fontsize)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()


### Vary decay rate

In [36]:
Bc_wave

array([4.90505218e-05, 4.90505218e-05, 6.99020296e-05, 6.99020296e-05,
       6.99020296e-05, 9.96175690e-05, 9.96175690e-05, 9.96175690e-05,
       1.41965264e-04, 1.41965264e-04, 1.41965264e-04, 1.41965264e-04,
       1.41965264e-04, 2.02315076e-04, 2.02315076e-04, 2.02315076e-04,
       2.02315076e-04, 2.02315076e-04, 2.02315076e-04, 2.88319755e-04])

In [39]:
"""vary decay rate. now time in minutes. length in microns"""
B0s = np.logspace(-6, -3, 40)
KD = 0.01#np.logspace(-2, -0.5, 5)
t_maxs = np.linspace(2000, 100, 20)
L = 100.0
n = 2000
hill_coefficient = 3
gammas = np.linspace(0.02, 0.2, 20)
D = 1.0    
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)
n_iters = 5
scale = 0.1


"""tissue"""
print('tissue')
Bc_wave = np.zeros(len(gammas))
Bc_wave_uncertainties = np.zeros(len(gammas))
for i in range(len(gammas)):
    print(f'{i + 1} of {len(gammas)}')
    Bc_wave[i], Bc_wave_uncertainties[i] = compute_wave_threshold(B0s, KD=KD, t_max=t_maxs[i], 
                       hill_coefficient=hill_coefficient, D=D, gamma=gammas[i], shape=shape, 
                       lb=lb, h=h, L=L, scale=scale, n_iters=n_iters, **kwargs)
      


gamma_sweep_dict = {'gammas': gammas, 'Bc_wave': Bc_wave, 'Bc_wave_uncertainties': Bc_wave_uncertainties}
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/gamma_sweep_scale0pt1_longer-time.pkl', 'wb') as f:
    pickle.dump(gamma_sweep_dict, f)    

tissue
1 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

2 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

3 of 20


...

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

19 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

20 of 20


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
A

In [47]:
gamma_sweep_dict = {'gammas': gammas, 'Bc_wave': Bc_wave, 'Bc_cell': Bc_cell, 'Bc_wave_uncertainties': Bc_wave_uncertainties, 'Bc_cell_uncertainties': Bc_cell_uncertainties}
with open(r'/home/brandon/Documents/Code/immunowave/data/initial-theory-numerics-comparison/gamma_sweep.pkl', 'wb') as f:
    pickle.dump(gamma_sweep_dict, f)

In [40]:
"""plot"""
gamma_theory = np.linspace(np.min(gammas), np.max(gammas), 1000)

# load the data
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/gamma_sweep_scale0pt1_longer-time.pkl', 'rb') as f:
    gamma_sweep_dict = pickle.load(f)
gammas = gamma_sweep_dict['gammas']
Bc_wave = gamma_sweep_dict['Bc_wave']

plt.figure(figsize=(8,7))
plt.plot(gammas, Bc_wave, marker='o', markersize=18, markerfacecolor='none', 
             markeredgewidth=4, linestyle='none', markeredgecolor=colors['wave'], label='wave numerics')
tissue_theory = 2 * (1 - (2 / (hill_coefficient + 1))) ** 0.5 * (KD * gamma_theory) ** (hill_coefficient / (hill_coefficient - 1)) * np.sqrt(D / gamma_theory)
plt.plot(gamma_theory, tissue_theory, '-', linewidth=linewidth, color=colors['wave'], label='wave theory')

plt.legend(fontsize=18)
plt.xscale("linear")
plt.yscale("log")
plt.ylabel(r"$B_{c}$ (a.u.)", fontsize=24)
plt.xlabel(r"loss rate, $\gamma$ (1/min)", fontsize=24)
plt.minorticks_off()
ax = plt.gca()
style_axes(ax)
plt.tight_layout()


## Plot everything including schematics

In [325]:
KD = 0.4
hill_coefficient = 3

u_vals = np.linspace(0, 1.1, 20)
v_vals = np.linspace(-0.4, 0.4, 20)
U, V = np.meshgrid(u_vals, v_vals)
dU = V
dV = -derivative_func(U, hill_coefficient, KD)
magnitude = np.sqrt(dU ** 2 + dV ** 2)
dU_norm = dU / magnitude
dV_norm = dV / magnitude

plt.figure()
#plt.quiver(U, V, dU_norm, dV_norm, magnitude)

A = np.linspace(0, 1.1, 10000)
V = np.cumsum(derivative_func(A[:-1], hill_coefficient, KD) * np.diff(A)[0])
E = 0
dAdx = (2 * (E - V)) ** 0.5

plt.plot(A[:-1], dAdx, '-', linewidth=linewidth, color='k')
plt.plot(A[:-1], -dAdx, '-', linewidth=linewidth, color='k')

# find E = V(u3), where u3 is the high fixed point
pol = [-1, 1, 0, -KD ** hill_coefficient, 0]
roots = np.roots(pol)
big_root = np.max(roots)
#A = np.linspace(0.75, 1.1, 10000)
#V = np.cumsum(derivative_func(A[:-1], hill_coefficient, KD) * np.diff(A)[0])
E = V[np.where(np.abs(A - big_root) == np.min(np.abs(A - big_root)))[0][0]]
dAdx = (2 * (E - V)) ** 0.5
plt.plot(A[:-1], dAdx, '-', linewidth=linewidth, color=colors['wave'])
plt.plot(A[:-1], -dAdx, '-', linewidth=linewidth, color=colors['wave'])


plt.plot(0, 0, 's', markersize=12, markeredgecolor='r', markeredgewidth=2, markerfacecolor='none')
plt.plot(roots[1], 0, 'o', markersize=12, markeredgecolor='r',  markeredgewidth=2, markerfacecolor='none')
plt.plot(roots[0], 0, 's', markersize=12, markeredgecolor='r',  markeredgewidth=2, markerfacecolor='none')

plt.xlabel('$u$', fontsize=fontsize)
plt.ylabel('$\partial_x u$', fontsize=fontsize)
plt.xlim([-0.1, 1.1])
#plt.ylim([-0.4, 0.4])
ax = style_axes(plt.gca())
plt.tight_layout()

In [315]:
roots

array([ 0.99899799,  0.0321436 , -0.03114159,  0.        ])

In [313]:
roots

array([ 0.99899799,  0.0321436 , -0.03114159,  0.        ])

In [144]:
plt.close('all')

In [145]:
#f, axs = plt.subplots(2, 3, figsize=(6.5, 4))
f, axs = plt.subplots(2, 3, figsize=(18.97, 10.79))

### Solution branches schematic

In [146]:
def derivative_func(A, hill_coefficient, KD):
    return A ** hill_coefficient / (KD ** hill_coefficient + A ** hill_coefficient) - A
    #return A * (A - 0.4) * (1 - A)

In [169]:
KD = 0.4
hill_coefficient = 4

ax = axs[0, 0]
ax.clear()

A = np.linspace(0, 1.1, 10000)
V = np.cumsum(derivative_func(A[:-1], hill_coefficient, KD) * np.diff(A)[0])
E = 0
dAdx = (2 * (E - V)) ** 0.5
max_dAdx = np.nanmax(dAdx)

ax.plot(A[:-1], dAdx, '-', linewidth=linewidth, color=(0.5, 0.5, 0.5))
ax.plot(A[:-1], -dAdx, '-', linewidth=linewidth, color=(0.5, 0.5, 0.5))

# find E = V(u3), where u3 is the high fixed point
pol = [-1, 1, 0, 0, -KD ** hill_coefficient, 0]
roots = np.roots(pol)
big_root = np.max(roots)
E = V[np.where(np.abs(A - big_root) == np.min(np.abs(A - big_root)))[0][0]]
dAdx = (2 * (E - V)) ** 0.5
ax.plot(A[:-1], dAdx, '-', linewidth=linewidth, color=colors['wave'])
ax.plot(A[:-1], -dAdx, '-', linewidth=linewidth, color=colors['wave'])


ax.plot(0, 0, 's', markersize=markersize, markeredgecolor='r', markeredgewidth=markeredgewidth, markerfacecolor='none')
ax.plot(roots[1], 0, 'o', markersize=markersize, markeredgecolor='r',  markeredgewidth=markeredgewidth, markerfacecolor='none')
ax.plot(roots[0], 0, 's', markersize=markersize, markeredgecolor='r',  markeredgewidth=markeredgewidth, markerfacecolor='none')

xline = np.linspace(-.1, 1.1, 5)
ax.plot(xline, max_dAdx * np.ones_like(xline), '--', linewidth=linewidth, color='k')
ax.plot(xline, -max_dAdx * np.ones_like(xline), '--', linewidth=linewidth, color='k')

ax.set_xlabel('$u$ (a.u.)', fontsize=fontsize)
ax.set_ylabel('$\partial_x u$ (a.u.)', fontsize=fontsize)
ax.set_title('steady state\nphase plane', fontsize=fontsize)
ax.set_xlim([-0.1, 1.1])
#ax.set_ylim([-1.5 * max_dAdx, 1.5 * max_dAdx])
ax = style_axes(ax, fontsize=fontsize)


### Wave threshold schematic

In [142]:
axs[0,1].clear()

In [148]:
KD = 0.4
t_max = 300#45
L = 3*100.0
n = 2000
hill_coefficient = 4
gamma = 1.0
D = 1.0    
shape = (n,)
lb = [0]
h = L / (n - 1)
kwargs = dict(dt0=1e-4, max_steps=100000000, atol=1e-3, rtol=1e-3)

"""below thresh"""
B0 = 1.22 * Bc_wave_theory(hill_coefficient, KD, gamma, D, a=1)
model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, gamma=gamma)
state = State4(
        A=iw.ScalarField(shape, lb, h, 0),
        B=iw.ScalarField(
            shape, lb, h, fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L/ 2, scale=0.5)
            ),
        )
        
solution = iw.solve(
        model,
        state,
        t0=0,
        t1=t_max,
        saveat=SaveAt(ts=jnp.linspace(0, t_max, 8)),
        **kwargs,
        )
        
        
A = solution.ys.A.values
x = np.linspace(0, L, n)
# n_plot_ts = 8
# plot_ts = np.linspace(0, len(A), n_plot_ts, dtype='int')
# plot_reds = np.linspace(0.5, colors['wave'][0], 8)
# plot_greens = np.linspace(0.5, colors['wave'][1], 8)
# plot_blues = np.linspace(0.5, colors['wave'][2], 8)

ax = axs[0, 1]
ax.clear()
# for i in range(n_plot_ts):
#     ax.plot(x, A[plot_ts[i]], '-', linewidth=linewidth, color=(plot_reds[i], plot_greens[i], plot_blues[i]))

ax.plot(x, A[-1], '-', linewidth=linewidth, color=(0.5, 0.5, 0.5))

ax.set_xlabel('space, $x$ (a.u.)', fontsize=fontsize)
ax.set_ylabel('$u$ (a.u.)', fontsize=fontsize)
#ax.set_ylim([0, 1])
#ax.set_xlim([145, 155])
ax.set_title(f'local response\n$I={B0:.3f}$', fontsize=fontsize)
ax.ticklabel_format(scilimits=(-3, 6))
ax = style_axes(ax, fontsize=fontsize)


"""above thresh"""
#t_max = 100#45#15
B0 = 1.23 * Bc_wave_theory(hill_coefficient, KD, gamma, D, a=1)
model = HillModel(KD=KD, n=hill_coefficient, η=1.0, ξ=0.0, λ=0.0, μ=0.0, gamma=gamma)
state = State4(
        A=iw.ScalarField(shape, lb, h, 0),
        B=iw.ScalarField(
            shape, lb, h, fn=lambda x: B0 * jax.scipy.stats.norm.pdf(x, loc=L/ 2, scale=0.5)
            ),
        )
        
solution = iw.solve(
        model,
        state,
        t0=0,
        t1=t_max,
        saveat=SaveAt(ts=jnp.linspace(0, t_max, t_max)),
        **kwargs,
        )
        
        
A = solution.ys.A.values
x = np.linspace(0, L, n)
n_plot_ts = 6#8
plot_ts = np.linspace(32, len(A), n_plot_ts, dtype='int')
plot_reds = np.linspace(0.5, colors['wave'][0], 8)
plot_greens = np.linspace(0.5, colors['wave'][1], 8)
plot_blues = np.linspace(0.5, colors['wave'][2], 8)

ax = axs[0, 2]
ax.clear()
for i in range(n_plot_ts):
    ax.plot(x, A[plot_ts[i]], '-', linewidth=linewidth, color=(plot_reds[i], plot_greens[i], plot_blues[i]))

ax.set_xlabel('space, $x$ (a.u.)', fontsize=fontsize)
ax.set_ylabel('$u$ (a.u.)', fontsize=fontsize)
ax.set_title(f'trigger wave\n$I={B0:.3f}$', fontsize=fontsize)
#ax.set_ylim([0, 1])
ax = style_axes(ax, fontsize=fontsize)


### comparison with numerics --- KD

In [149]:
def Bc_wave_semi_analytic(hill_coefficient, KD):
    pol = [-1, 1, 0, 0, -KD ** hill_coefficient, 0]
    roots = np.roots(pol)
    u2 = roots[1]
    #u2 = KD ** (hill_coefficient / (hill_coefficient - 1))
    u_arr = np.linspace(0, u2, 10000)
    du = np.diff(u_arr)[0]
    integrand = u_arr ** hill_coefficient / (KD ** hill_coefficient + u_arr ** hill_coefficient) - u_arr
    integral_term = np.sum(integrand * du)
    
    #Bc_wave = (-2 * (integral_term - KD ** (2 * hill_coefficient / (hill_coefficient - 1)) / 2)) ** (1 / 2)
    Bc_wave = (-2 * (integral_term)) ** (1 / 2)
    
    return Bc_wave

In [168]:
"""plot"""
# load the data
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/KD_sweep_wave_v3_scale0pt1.pkl', 'rb') as f:
    KD_sweep_dict = pickle.load(f)
KDs = KD_sweep_dict['KDs']
Bc_wave = KD_sweep_dict['Bc_wave']
# Bc_wave = Bc_wave[KDs < 0.18]
# KDs = KDs[KDs < 0.18]
hill_coefficient = 4

ax = axs[1, 0]
ax.clear()
KDs_theory = np.logspace(np.log10(np.min(KDs)), np.log10(np.max(KDs)), 1000)
tissue_theory = 2 * (1 - (2 / (hill_coefficient + 1))) ** 0.5 * KDs_theory ** (hill_coefficient / (hill_coefficient - 1))
ax.plot(KDs_theory, tissue_theory, '-', linewidth=linewidth, color=colors['wave'], label='analytic')


ax.plot(KDs, Bc_wave, marker='o', markersize=markersize, markerfacecolor='none', 
            markeredgewidth=markeredgewidth, linestyle='none', markeredgecolor=colors['wave'], label='numerics')


# tissue_semi_analytic = np.zeros_like(KDs_theory)
# for i in range(len(tissue_semi_analytic)):
#     tissue_semi_analytic[i] = Bc_wave_semi_analytic(hill_coefficient, KDs_theory[i])
# ax.plot(KDs_theory, 2 * tissue_semi_analytic, '--', linewidth=linewidth, color=colors['cell'], label='semi-analytic')

ax.legend(fontsize=fontsize)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel("concentration scale \nof positive feedback, $K_D$ (a.u.)", fontsize=fontsize)
ax.set_ylim([1e-3, 5e-1])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)


<Axes: xlabel='concentration scale \nof positive feedback, $K_D$ (a.u.)', ylabel='$I^{wave}_{c}$ (a.u.)'>

### comparison with numerics--- vary hill coeff

In [164]:
"""plot"""
ns = np.linspace(2, 5, 100)
KD = 0.01

# load the data
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/hill_coefficient_sweep_scale0pt1.pkl', 'rb') as f:
    hill_coeff_sweep_dict = pickle.load(f)
hill_coefficients = hill_coeff_sweep_dict['hill_coefficients']
Bc_wave = hill_coeff_sweep_dict['Bc_wave']

ax = axs[1, 1]
ax.clear()

tissue_theory = 2 * (1 - (2 / (ns + 1))) ** 0.5 * KD ** (ns / (ns - 1))
ax.plot(ns, tissue_theory, '-', linewidth=linewidth, color=colors['wave'], label='wave theory')

ax.plot(hill_coefficients, Bc_wave, marker='o', markersize=markersize, markerfacecolor='none', 
             markeredgewidth=markeredgewidth, linestyle='none', markeredgecolor=colors['wave'], label='wave numerics')



ax.set_xscale("linear")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel("hill coefficient, $n$", fontsize=fontsize)
ax.set_ylim([5e-5, 1e-2])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)


<Axes: xlabel='hill coefficient, $n$', ylabel='$I^{wave}_{c}$ (a.u.)'>

### comparision with numerics---vary gamma

In [161]:
"""plot"""
KD = 0.01

# load the data
with open(r'/home/brandon/Documents/Code/immunowave/data/corrected_theory_numerics_comparison/wave/gamma_sweep_scale0pt1_longer-time.pkl', 'rb') as f:
    gamma_sweep_dict = pickle.load(f)
gammas = gamma_sweep_dict['gammas']
Bc_wave = gamma_sweep_dict['Bc_wave']

gamma_theory = np.linspace(np.min(gammas), np.max(gammas), 1000)
hill_coefficient = 3

ax = axs[1,2]
ax.clear()

tissue_theory = 2 * (1 - (2 / (hill_coefficient + 1))) ** 0.5 * (KD * gamma_theory) ** (hill_coefficient / (hill_coefficient - 1)) * np.sqrt(D / gamma_theory)
ax.plot(gamma_theory, tissue_theory, '-', linewidth=linewidth, color=colors['wave'], label='wave theory')

ax.plot(gammas, Bc_wave, marker='o', markersize=markersize, markerfacecolor='none', 
             markeredgewidth=markeredgewidth, linestyle='none', markeredgecolor=colors['wave'], label='wave numerics')

ax.set_xscale("linear")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel(r"decay rate, $\gamma$ (1/min)", fontsize=fontsize)
ax.set_ylim([1e-5, 1e-3])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)


<Axes: xlabel='decay rate, $\\gamma$ (1/min)', ylabel='$I^{wave}_{c}$ (a.u.)'>

In [158]:
plt.gcf().tight_layout()

In [174]:
f = plt.gcf()

In [176]:
f.get_figheight()

10.79

In [175]:
f.get_figwidth()

18.97

In [177]:
plt.figure(figsize=(18.97, 10.79))

<Figure size 1897x1079 with 0 Axes>

In [173]:
plt.savefig(r'/home/brandon/Documents/Code/immunowave/plots/2025-04-15_numerics_comparision_fig-wave_v2.pdf')

## testing solution plot

In [182]:
θ = 0.4
u = np.linspace(0, 1.5, 10000)
I = np.sqrt(-2 * (-u ** 4 / 4 + (1 + θ) * u ** 3 / 3 - θ * u **2 / 2))
plt.figure()
plt.plot(u, I, 'k-', linewidth=linewidth)


$u' = v$

$v' = -u(u-\theta)(1-u)$

In [248]:
θ = 0.4
u = np.linspace(0, 1.1, 10000)
v = np.sqrt(-2 * (-u ** 4 / 4 + (1 + θ) * u ** 3 / 3 - θ * u **2 / 2))

u_vals = np.linspace(0, 1.1, 20)
v_vals = np.linspace(-0.25, 0.25, 20)
U, V = np.meshgrid(u_vals, v_vals)
dU = V
dV = -U * (U - θ) * (1 - U)
magnitude = np.sqrt(dU ** 2 + dV ** 2)
dU_norm = dU / magnitude
dV_norm = dV / magnitude

plt.figure()
plt.quiver(U, V, dU_norm, dV_norm, magnitude)
plt.plot(u, v, 'r-', linewidth=linewidth)
plt.plot(u, -v, 'r-', linewidth=linewidth)

u = np.linspace(0, 1.25, 10000)
V = -u **4 / 4 + (1 + θ) * u **3 / 3 - θ * u ** 2 / 2
E = -1/4 + (1 + θ) / 3 - θ / 2
v = np.sqrt(2 * (E - V))
plt.plot(u, v, 'r-', linewidth=linewidth)
plt.plot(u, -v, 'r-', linewidth=linewidth)

plt.plot(0, 0, 's', markersize=12, markeredgecolor='b', markeredgewidth=2, markerfacecolor='none')
plt.plot(θ, 0, 'o', markersize=12, markeredgecolor='b',  markeredgewidth=2, markerfacecolor='none')
plt.plot(1, 0, 's', markersize=12, markeredgecolor='b',  markeredgewidth=2, markerfacecolor='none')

plt.xlabel('$u$', fontsize=fontsize)
plt.ylabel('$\partial_x u$', fontsize=fontsize)
plt.xlim([-0.1, 1.1])
#plt.ylim([-0.4, 0.4])
ax = style_axes(plt.gca())
plt.tight_layout()

In [216]:
plt.figure()
plt.plot(u, (-u ** 4 / 4 + (1 + θ) * u ** 3 / 3 - θ * u **2 / 2))

In [237]:
u = np.linspace(0, 1.25, 10000)

f = -u **3 + (1 + θ) * u **2 - θ * u
plt.figure()
plt.plot(u, f)

In [230]:
V = -u **4 / 4 + (1 + θ) * u **3 / 3 - θ * u ** 2 / 2
plt.figure()
plt.plot(u, V)

In [236]:
E = -1/4 + (1 + θ) / 3 - θ / 2
E

0.016666666666666607

In [234]:
v = np.sqrt(2 * (E - V))

In [235]:
plt.figure()
plt.plot(u, v)
plt.plot(u, -v)

## phase plot of traveling wave solutiosn

In [267]:
θ = 0.4
c = 0.2

u_vals = np.linspace(0, 1.1, 20)
v_vals = np.linspace(-0.25, 0.25, 20)
U, V = np.meshgrid(u_vals, v_vals)
dU = V
dV = -c * V -U * (U - θ) * (1 - U)
magnitude = np.sqrt(dU ** 2 + dV ** 2)
dU_norm = dU / magnitude
dV_norm = dV / magnitude

plt.figure()
plt.quiver(U, V, dU_norm, dV_norm, magnitude)

u = np.linspace(0, 1.1, 10000)
v = np.sqrt(-2 * (-u ** 4 / 4 + (1 + θ) * u ** 3 / 3 - θ * u **2 / 2))

plt.plot(u, v, 'r-', linewidth=linewidth)
plt.plot(u, -v, 'r-', linewidth=linewidth)

u = np.linspace(0, 1.25, 10000)
V = -u **4 / 4 + (1 + θ) * u **3 / 3 - θ * u ** 2 / 2
E = -1/4 + (1 + θ) / 3 - θ / 2
v = np.sqrt(2 * (E - V))
plt.plot(u, v, 'r-', linewidth=linewidth)
plt.plot(u, -v, 'r-', linewidth=linewidth)

plt.plot(0, 0, 's', markersize=12, markeredgecolor='b', markeredgewidth=2, markerfacecolor='none')
plt.plot(θ, 0, 'o', markersize=12, markeredgecolor='b',  markeredgewidth=2, markerfacecolor='none')
plt.plot(1, 0, 's', markersize=12, markeredgecolor='b',  markeredgewidth=2, markerfacecolor='none')

plt.xlabel('$u$', fontsize=fontsize)
plt.ylabel('$\partial_x u$', fontsize=fontsize)
plt.xlim([-0.1, 1.1])
#plt.ylim([-0.4, 0.4])
ax = style_axes(plt.gca())
plt.tight_layout()

In [44]:
0.5 ** (3/4)

0.5946035575013605